<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# Cityscapes Semantic Segmentation — Benchmark (Akida 2)

Loads the converted `.fbz` segmentation model, evaluates mIoU on Akida (software backend if no hardware is attached), and — if a physical Akida 2 FPGA board is available — benchmarks latency.


## Setup


In [ ]:
import torch
import akida

from brainchip_utils.hardware_utils import AKIDA_CLOCKS_HZ

DATA_PATH = './data/cityscapes'
MODEL_FBZ = './pretrained_models/unetv4akida_segmentation_i8_w8_a8.fbz'


## Load the Akida model


In [ ]:
ak_model = akida.Model(MODEL_FBZ)
ak_model.summary()


## Device

`get_akida_device` returns `None` when no compatible hardware is present, in which case the software backend is used instead (evaluation still works; latency benchmarking below is skipped).


In [ ]:
from brainchip_utils.hardware_utils import get_akida_device

device = get_akida_device(target_version=ak_model.ip_version)
print('Akida hardware found:', device) if device is not None else print('No hardware found — using software backend')


## Evaluation

mIoU over a subsample of the Cityscapes val split (the full 500-image split, mosaic tiling only, takes about an hour on the software backend — see README.md).


In [ ]:
from segmentation_data import _image_mask_paths
from segmentation_eval import eval_cityscapes_set

img_paths, mask_paths = _image_mask_paths(DATA_PATH, 'val')

def akida_predict(patch):
    out = ak_model.predict(patch.permute(0, 2, 3, 1).numpy().astype('uint8'))
    return torch.from_numpy(out).permute(0, 3, 1, 2)

n_eval = 20
iou, miou, gt_pct = eval_cityscapes_set(
    akida_predict, img_paths, mask_paths, use_hann=False, max_images=n_eval, akida_uint8=True)
print(f'Akida mIoU (n={n_eval}, mosaic): {miou:.4f}')


## Activation Sparsity


In [ ]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity
from segmentation_data import get_samples

calib_samples = get_samples(DATA_PATH, num_samples=1024)
sparsity_dict = compute_sparsity(ak_model, samples=calib_samples)
pretty_print_sparsity(sparsity_dict)


## Hardware Benchmark

Requires a physical Akida 2 FPGA platform with a connected board. Unlike the eyetracking example, this model is a plain feedforward network (no streaming/FIFO state), so the standard batched benchmark helpers apply directly, one 384x384 tile per inference — see `segmentation_benchmark.py` for the full per-layer benchmark and plots.


In [ ]:
from brainchip_utils.hardware_utils import full_model_benchmark, get_mapping_stats

if device is not None:
    ak_model.map(device, mode=akida.MapMode.Minimal)
    measured_clock = AKIDA_CLOCKS_HZ['AKIDA2_FPGA']
    projected_clock = AKIDA_CLOCKS_HZ['AKD2500']
    res = full_model_benchmark(ak_model, device, calib_samples[:100],
                                map_mode=akida.MapMode.Minimal, clock_freq=measured_clock)
    projected_ms = res['mean_inf_clk'] / projected_clock * 1000
    print(f'Measured @ {measured_clock/1e6:.0f} MHz: {res["mean_clk_ms"]:.3f} ms/tile')
    print(f'Projected @ {projected_clock/1e6:.0f} MHz: {projected_ms:.3f} ms/tile')
else:
    print('Hardware not available — skipping benchmark')


## Summary


In [ ]:
print(f'Akida mIoU: {miou:.4f}')
print(f'Mean activation sparsity: {sum(sparsity_dict.values()) / len(sparsity_dict) * 100:.2f}%')
